# Experiment 18 - Native-Categorical CatBoost

This experiment gives CatBoost categorical features directly instead of one-hot encoding them.

The goal is to test whether CatBoost's native categorical handling can outperform the current one-hot-based models.

Current local best: **0.941815**

Current Kaggle best from our submissions: **0.941680**

Leaderboard target: **0.946740**

In [1]:
from pathlib import Path
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from catboost import CatBoostClassifier

PROJECT_ROOT = Path(r'C:\Users\aakif\Documents\DataCompetition')
TRAIN_PATH = PROJECT_ROOT / 'data' / 'train.csv'

train = pd.read_csv(TRAIN_PATH)

X = train.drop(columns=['Will_Buy_EV', 'id']).copy()
y = train['Will_Buy_EV'].map({'No': 0, 'Yes': 1})

X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

categorical_features = X_train.select_dtypes(exclude=['number']).columns.tolist()
categorical_indices = [X_train.columns.get_loc(column) for column in categorical_features]

for column in categorical_features:
    X_train[column] = X_train[column].fillna('Missing').astype(str)
    X_valid[column] = X_valid[column].fillna('Missing').astype(str)

print('Training rows:', len(X_train))
print('Validation rows:', len(X_valid))
print('Total features:', X_train.shape[1])
print('Categorical features:', len(categorical_features))
print('Categorical columns:', categorical_features)

Training rows: 534932
Validation rows: 133733
Total features: 13
Categorical features: 6
Categorical columns: ['Gender', 'City_Type', 'Current_Car_Type', 'Home_Charging_Possible', 'Subsidy_Available', 'Range_Anxiety_Level']


In [2]:
configs = {
    'CatBoost_Native_1': {
        'iterations': 1200,
        'depth': 6,
        'learning_rate': 0.04,
        'l2_leaf_reg': 5,
        'random_strength': 1.0,
        'bagging_temperature': 1.0
    },
    'CatBoost_Native_2': {
        'iterations': 1500,
        'depth': 7,
        'learning_rate': 0.03,
        'l2_leaf_reg': 5,
        'random_strength': 0.5,
        'bagging_temperature': 1.0
    },
    'CatBoost_Native_3': {
        'iterations': 1500,
        'depth': 8,
        'learning_rate': 0.03,
        'l2_leaf_reg': 8,
        'random_strength': 0.5,
        'bagging_temperature': 1.0
    }
}

predictions = {}
results = []

for name, params in configs.items():
    print(f'\nTraining {name}...')

    model = CatBoostClassifier(
        **params,
        loss_function='Logloss',
        eval_metric='AUC',
        random_seed=42,
        verbose=False,
        thread_count=-1,
        allow_writing_files=False
    )

    model.fit(
        X_train,
        y_train,
        cat_features=categorical_indices,
        eval_set=(X_valid, y_valid),
        use_best_model=True,
        early_stopping_rounds=100
    )

    pred = model.predict_proba(X_valid)[:, 1]
    score = roc_auc_score(y_valid, pred)

    predictions[name] = pred
    results.append({
        'Model': name,
        'ROC-AUC': score,
        'Best Iteration': model.get_best_iteration()
    })

results_df = pd.DataFrame(results).sort_values('ROC-AUC', ascending=False).reset_index(drop=True)

print('\n' + '=' * 70)
print('EXPERIMENT 18 RESULTS')
print('=' * 70)
print(results_df.to_string(index=False))


Training CatBoost_Native_1...

Training CatBoost_Native_2...

Training CatBoost_Native_3...

EXPERIMENT 18 RESULTS
            Model  ROC-AUC  Best Iteration
CatBoost_Native_2 0.941464            1499
CatBoost_Native_1 0.941463            1197
CatBoost_Native_3 0.941401            1499


In [3]:
previous_best = 0.941815

best_model_name = results_df.iloc[0]['Model']
best_score = results_df.iloc[0]['ROC-AUC']

print(f'Previous local best: {previous_best:.6f}')
print(f'Best Experiment 18 model: {best_model_name}')
print(f'Best Experiment 18 ROC-AUC: {best_score:.6f}')
print(f'Difference vs previous best: {best_score - previous_best:+.6f}')

if best_score > previous_best:
    print('\nNEW LOCAL BEST MODEL')
else:
    print('\nNo Experiment 18 model beat the current local best.')

Previous local best: 0.941815
Best Experiment 18 model: CatBoost_Native_2
Best Experiment 18 ROC-AUC: 0.941464
Difference vs previous best: -0.000351

No Experiment 18 model beat the current local best.
